# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器 (History & LMR Drive-Chain Phase 6 @ 100k nodes)

本 Notebook 用於執行 **Phase 6：歷史動力鏈與 LMR 聯動體系（History & LMR Drive-Chain Tuning @ 100k nodes）** 雲端 SPSA 調參與深度 SPRT 對抗驗收。

### 🎯 Phase 6 核心調參六軸（全部為運行時免重編譯軸）
1. `HISTORY_BONUS_CAP`（當前基線 2054，範圍 [800, 3000]，步長 100）：歷史表加分上限天花板（E63 放寬新基線）。
2. `HISTORY_BONUS_SCALE`（當前基線 120，範圍 [40, 240]，步長 10）：安靜步截斷歷史加分斜率（`bonus = min(SCALE * depth, CAP)`）。
3. `LMR_HISTORY_SCALE`（當前基線 204，範圍 [50, 300]，步長 25）：歷史評分轉化為 LMR 減步的斜率（直接咬合上述兩軸，折算減深寬減）。
4. `LMR_BASE_OFFSET`（當前基線 437，範圍 [250, 700]，步長 25）：全局 LMR 減深基礎偏移量。
5. `LMR_MOVECOUNT_FACTOR`（當前基線 35，範圍 [0, 100]，步長 4）：晚期走步隨計數遞增的反向阻尼速度。
6. `HISTORY_MALUS_CAP`（當前基線 1600，範圍 [800, 3000]，步長 100）：歷史失敗走步懲罰扣分上限。

### ⚡ 調參架構（100k 高深度視野 + 雙倍抗噪阻尼，複製 Phase 5 成功方程式）
* **L1 雲端 SPSA 探索**：60 輪 × 300 局 = **18,000 局 @ 100,000 nodes**（`inprocess` 免重複 JIT 高速對抗）
* **SPSA 阻尼調優**：**`--spsa-a-ratio 0.2`**（放大穩定常數 $A$，對沖 300 局單輪訊號翻倍帶來的步伐過大問題，確保後期平穩收斂）
* **Polyak-Ruppert 平滑取樣**：實證黃金比例 **`--tail 0.3`**（後 18 輪 / 5,400 局加權平均，徹底消除單輪梯度隨機噪聲）
* **L2 深度獨立驗收**：600~1,000 局 @ 300,000 nodes，主開局庫 `data/openings.epd`，SPRT(0.0, 3.0)


In [1]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.16


In [6]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
!git reset --hard
!git clean -fd
!rm -f tournament_analysis/l2_verify_history_lmr_p6_deep_tail03.json
!git pull origin {branch}
!pip install -q numba chess numpy


/content/chess_numba
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
HEAD is now at acd97de feat(tune): checkpoint Phase 6 progress at 23 iterations (6.9k games) for seamless resume
Removing tune_search/candidate_history_lmr_p6_deep.json
Removing tune_search/candidate_history_lmr_p6_tail03_linear.json
Removing tune_search/candidate_history_lmr_p6_tail03_uniform.json
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 6.23 KiB | 6.23 MiB/s, done.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
   acd97de..a8c23d6  feat-initial-bitboard-setup -> origin/feat-initial-bitboard-setup
Updating acd97de..a8c23d6
Fast-forward
 tune_colab_runner_p6.ipynb                     | 1576 +++++++----------

In [7]:
# 3. 測試環境與驗證 Phase 6 基準常數
import sys, json
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

phase6_params = [
    "HISTORY_BONUS_CAP",
    "HISTORY_BONUS_SCALE",
    "LMR_HISTORY_SCALE",
    "LMR_BASE_OFFSET",
    "LMR_MOVECOUNT_FACTOR",
    "HISTORY_MALUS_CAP"
]

print("🔍 Phase 6 歷史動力鏈與 LMR 聯動六軸基準值（Baseline）：")
baseline = get_default_params(phase6_params)
print(json.dumps(baseline, indent=2))

with open("tune_search/baseline_history_lmr_p6.json", "r", encoding="utf-8") as f:
    saved_baseline = json.load(f)

assert baseline == saved_baseline, f"Baseline mismatch! {baseline} vs {saved_baseline}"
print("✅ Phase 6 基準檔 tune_search/baseline_history_lmr_p6.json 驗證一致！")


🔍 Phase 6 歷史動力鏈與 LMR 聯動六軸基準值（Baseline）：
{
  "HISTORY_BONUS_CAP": 2054,
  "HISTORY_BONUS_SCALE": 120,
  "LMR_HISTORY_SCALE": 204,
  "LMR_BASE_OFFSET": 437,
  "LMR_MOVECOUNT_FACTOR": 35,
  "HISTORY_MALUS_CAP": 1600
}
✅ Phase 6 基準檔 tune_search/baseline_history_lmr_p6.json 驗證一致！


In [8]:
# 4. 啟動 L1 SPSA Phase 6 歷史與 LMR 六軸雲端對抗調參（目標 60 輪 x 300 局 = 18,000 局 @ 100k nodes）
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/history_lmr_p6_100k_l1.jsonl"
state_file = "tune_search/states/history_lmr_p6_100k_l1.json"
total_campaign_iters = 60

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數（支援斷點續跑）
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params HISTORY_BONUS_CAP,HISTORY_BONUS_SCALE,LMR_HISTORY_SCALE,LMR_BASE_OFFSET,LMR_MOVECOUNT_FACTOR,HISTORY_MALUS_CAP \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 300 \
      --nodes 100000 \
      --spsa-a-ratio 0.2 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ Phase 6 的 60 輪調參已全部完成！請直接執行步驟 5 產出候選參數。\n")


偵測到 Colab CPU 核心數: 8，調參並行度設為: 7
🔄 斷點續跑模式：已完成 23/60 輪，本次接續執行剩餘 37 輪！
[resume] iteration=23 from tune_search/states/history_lmr_p6_100k_l1.json (inprocess)
[inprocess] openings=500 nodes=100000 concurrency=7 tt=16MB
[pool] allocate worker pair 1/7 (tt=16MB)...
[pool] allocate worker pair 2/7 (tt=16MB)...
[pool] allocate worker pair 3/7 (tt=16MB)...
[pool] allocate worker pair 4/7 (tt=16MB)...
[pool] allocate worker pair 5/7 (tt=16MB)...
[pool] allocate worker pair 6/7 (tt=16MB)...
[pool] allocate worker pair 7/7 (tt=16MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 1.0s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 2/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 3/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[wa

In [9]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後 30% 平均（tail 0.3）產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/history_lmr_p6_100k_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_history_lmr_p6_deep.json
!python tune_search/summarize_run.py tune_search/logs/history_lmr_p6_100k_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_history_lmr_p6_tail03_uniform.json
!python tune_search/summarize_run.py tune_search/logs/history_lmr_p6_100k_l1.jsonl --tail 0.3 --weight-mode linear --out tune_search/candidate_history_lmr_p6_tail03_linear.json
print("\n=== Phase 6 等權最佳候選參數 JSON (tail 0.3 uniform) ===")
!cat tune_search/candidate_history_lmr_p6_deep.json
print("\n=== Phase 6 線性加權候選參數 JSON (tail 0.3 linear) ===")
!cat tune_search/candidate_history_lmr_p6_tail03_linear.json


iterations=60 games=18000 mean_score=0.5034
score sd observed=0.0213 noise-only expected=0.0203 ratio=1.05
  -> ratio ~1: batch scores are indistinguishable from sampling noise; this run carries (almost) no gradient signal. Use more games per batch / more batches before trusting any parameter movement.

param                             default      last  tail_avg moved%rng  drift_z
HISTORY_BONUS_CAP                    2054    2045.0    2026.1     -0.41    -0.08  (noise)
HISTORY_BONUS_SCALE                   120     136.0     137.0      8.02     1.45  (noise)
LMR_HISTORY_SCALE                     204     200.4     200.0     -1.45    -0.13  (noise)
LMR_BASE_OFFSET                       437     448.1     448.7      2.46     0.40  (noise)
LMR_MOVECOUNT_FACTOR                   35      29.1      29.3     -5.89    -1.33  (noise)
HISTORY_MALUS_CAP                    1600    1784.3    1788.1      8.38     1.66  (noise)

averaged candidate (last 18 iterations, weight_mode=uniform):
--param HIS

In [10]:
# 6. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600~1000 局，主開局庫獨立驗證）
# 對照基準明確指定為 Phase 6 Baseline (tune_search/baseline_history_lmr_p6.json)
import os

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"啟動 Phase 6 L2 深度驗收，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_history_lmr_p6_deep.json \
  --baseline-json tune_search/baseline_history_lmr_p6.json \
  --nodes 300000 \
  --games 1000 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_history_lmr_p6_deep_tail03.json


啟動 Phase 6 L2 深度驗收，使用並行度: 7
search_param_match nodes=300000 games<=1000 concurrency=7 openings=2000 sprt=(0.0, 3.0)
baseline={'HISTORY_BONUS_CAP': 2054, 'HISTORY_BONUS_SCALE': 120, 'LMR_HISTORY_SCALE': 204, 'LMR_BASE_OFFSET': 437, 'LMR_MOVECOUNT_FACTOR': 35, 'HISTORY_MALUS_CAP': 1600}
candidate={'HISTORY_BONUS_CAP': 2026, 'HISTORY_BONUS_SCALE': 137, 'LMR_HISTORY_SCALE': 200, 'LMR_BASE_OFFSET': 449, 'LMR_MOVECOUNT_FACTOR': 29, 'HISTORY_MALUS_CAP': 1788}
[pool] allocate worker pair 1/7 (tt=16MB)...
[pool] allocate worker pair 2/7 (tt=16MB)...
[pool] allocate worker pair 3/7 (tt=16MB)...
[pool] allocate worker pair 4/7 (tt=16MB)...
[pool] allocate worker pair 5/7 (tt=16MB)...
[pool] allocate worker pair 6/7 (tt=16MB)...
[pool] allocate worker pair 7/7 (tt=16MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 0.9s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first